In [1]:
import os
import json
import time
from pprint import pprint
from datasets import Dataset

import torch
from sentence_transformers import SentenceTransformer
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from sentence_transformers.util import cos_sim
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers import SentenceTransformerTrainingArguments
from sentence_transformers.training_args import BatchSamplers
from sentence_transformers import SentenceTransformerTrainer

/Users/bowie/anaconda3/envs/good/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
with open("/Users/bowie/Documents/pycharmfile/embedding_model_exp/data/ft_val_dataset.json", "r", encoding="utf-8") as f:
    eval_content = json.loads(f.read())

corpus, queries, relevant_docs = eval_content['corpus'], eval_content['queries'], eval_content['relevant_docs']

In [3]:
# /Users/bowie/Documents/pycharmfile/embedding_model_exp/data/ft_val_dataset.json
with open("/Users/bowie/Documents/pycharmfile/embedding_model_exp/data/ft_train_dataset.json", "r", encoding="utf-8") as f:
    train_content = json.loads(f.read())


In [4]:
train_anchor, train_positive = [], []
for query_id, context_id in train_content['relevant_docs'].items():
    train_anchor.append(train_content['queries'][query_id])
    train_positive.append(train_content['corpus'][context_id[0]])
    # break

# print(train_anchor, train_positive)

In [5]:
train_dataset = Dataset.from_dict({"positive": train_positive, "anchor": train_anchor})



In [6]:
print(train_dataset)
pprint(train_dataset[0:5])

Dataset({
    features: ['positive', 'anchor'],
    num_rows: 178
})
{'anchor': ['根据上下文，你认为今年A股半导体行业上市公司的半年度业绩受到了哪些因素的影响？',
            '根据上述信息，请问哪些半导体上市公司预计业绩首次亏损？',
            'What is the reason for the decline in performance of chip design '
            'companies in the first half of the year in the context of the '
            'sluggish consumer electronics market?',
            '根据公司表示的情况，上半年公司面临哪些挑战导致毛利率下降？',
            '根据上述信息，你认为造成兆易创新上半年归母净利润下降超过七成的主要原因是什么？'],
 'positive': ['受半导体行业周期“磨底”、消费电子市场需求恢复缓慢等影响，今年A股半导体行业上市公司半年度业绩预告显示，归母净利润普遍同比下滑，IC设计、封测等环节成为“重灾区”， '
              '。环比来看，部分头部企业第二季度业绩已经企稳复苏，盈利环比增长，人工智能、汽车电子、电网等板块贡献业绩，有公司表示下半年将企稳增长。',
              '据Choice金融终端统计，目前超过30家半导体上市公司披露业绩预告，其中，通富微电、汇顶科技、士兰微、上海贝岭、中晶科技、大为股份等公司业绩预计首亏，博通集成预亏增加，韦尔股份、瑞芯微、华天科技等公司最大降幅超过90%。相比之下，北方华创、中微公司等头部企业翻倍增长。\n'
              '\n'
              '\u3000\u3000设计企业：',
              '加速去库存\n'
              '\n'
              '\u3000\u3000'
              '由于终端消费电子市场低迷，芯片设计企业上半年业绩同比普遍预降，但随着去库存推进，部分企

In [7]:
model = SentenceTransformer("/Users/bowie/Documents/muti-model/bge-base-zh-v1.5", device="cuda:0" if torch.cuda.is_available() else "cpu")


In [8]:
model_name = 'bge-base-zh-v1.5'

In [9]:
evaluator = InformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    name=f"{model_name}",
    score_functions={"cosine": cos_sim}
)

In [10]:
train_loss = MultipleNegativesRankingLoss(model)

In [15]:
"""
{
'eval_bge-base-zh-v1.5_cosine_accuracy@1': 0.7014925373134329, 'eval_bge-base-zh-v1.5_cosine_accuracy@3': 0.8208955223880597, 
'eval_bge-base-zh-v1.5_cosine_accuracy@5': 0.8507462686567164, 'eval_bge-base-zh-v1.5_cosine_accuracy@10': 0.8656716417910447, 
'eval_bge-base-zh-v1.5_cosine_precision@1': 0.7014925373134329, 'eval_bge-base-zh-v1.5_cosine_precision@3': 0.2736318407960198, 
'eval_bge-base-zh-v1.5_cosine_precision@5': 0.17014925373134326, 'eval_bge-base-zh-v1.5_cosine_precision@10': 0.08656716417910446, 
'eval_bge-base-zh-v1.5_cosine_recall@1': 0.7014925373134329, 'eval_bge-base-zh-v1.5_cosine_recall@3': 0.8208955223880597, 
'eval_bge-base-zh-v1.5_cosine_recall@5': 0.8507462686567164, 'eval_bge-base-zh-v1.5_cosine_recall@10': 0.8656716417910447, 
'eval_bge-base-zh-v1.5_cosine_ndcg@10': 0.7862251404707558, 'eval_bge-base-zh-v1.5_cosine_mrr@10': 0.7601368159203978, 
'eval_bge-base-zh-v1.5_cosine_map@100': 0.7629992332622932, 
'eval_runtime': 4.877, 'eval_samples_per_second': 0.0, 'eval_steps_per_second': 0.0, 'epoch': 2.93
}
"""

args = SentenceTransformerTrainingArguments(
    output_dir=f"ft_{model_name}",  # output directory and hugging face model ID 输出目录
    num_train_epochs=3,  # number of epochs 训练的总轮数
    per_device_train_batch_size=4,  # train batch size 每个设备上的训练批次大小
    gradient_accumulation_steps=2,  # for a global batch size of 512 梯度累积的步数
    per_device_eval_batch_size=4,  # evaluation batch size 每个设备上的评估批次大小
    warmup_ratio=0.1,  # warmup ratio 学习率预热的比例
    learning_rate=2e-5,  # learning rate, 2e-5 is a good value 学习率
    lr_scheduler_type="cosine",  # use constant learning rate scheduler 学习率调度器类型
    # optim="adamw_torch_fused",  # use fused adamw optimizer 优化器类型
    # tf32=True,  # use tf32 precision tf32 requires Ampere or a newer GPU arch, cuda>=11 and torch>=1.7
    bf16=True,  # use bf16 precision
    batch_sampler=BatchSamplers.NO_DUPLICATES,  # 禁止重复采样相同的训练数据
    eval_strategy="epoch",  # evaluate after each epoch
    save_strategy="epoch",  # save after each epoch 保存模型的策略（"steps" 或 "epoch"）
    logging_steps=10,  # log every 10 steps 日志打印的步数
    save_total_limit=3,  # save only the last 3 models 保存的模型文件总数限制
    load_best_model_at_end=True,  # load the best model when training ends 是否在训练结束时加载最优模型
    # metric_for_best_model=f"eval_{model_name}_cosine_ndcg@10",  # Optimizing for the best ndcg@10 score 用于评估最优模型的指标 这玩意的中间得和evaluator的名字一样
    # eval_bge-base-zh-v1.5_cosine_ndcg@10
    metric_for_best_model=f"eval_bge-base-zh-v1.5_cosine_ndcg@10",  # eval_bge-base-zh-v1.5-model_evaluation_cosine_ndcg@10
    report_to='tensorboard'
)

In [16]:
start_time = time.time()

In [17]:
trainer = SentenceTransformerTrainer(
    model=model,    # the model to train
    args=args,      # training arguments
    train_dataset=train_dataset.select_columns(
        ["positive", "anchor"]
    ),  # training dataset
    loss=train_loss,
    evaluator=evaluator
)

In [18]:
trainer.train()
trainer.save_model()
print(f"cost time: {time.time() - start_time:.2f}s")

Column 'anchor' is at index 1, whereas a column with this name is usually expected at index 0. Note that the column order can be important for some losses, e.g. MultipleNegativesRankingLoss will always consider the first column as the anchor and the second as the positive, regardless of the dataset column names. Consider renaming the columns to match the expected order, e.g.:
dataset = dataset.select_columns(['anchor', 'positive', 'negative'])
                                               
 33%|███▎      | 22/66 [01:49<01:05,  1.49s/it]

{'loss': 0.2564, 'grad_norm': 4.320690155029297, 'learning_rate': 1.9872683547213446e-05, 'epoch': 0.44}


                                               
 33%|███▎      | 22/66 [02:03<01:05,  1.49s/it]

{'loss': 0.2166, 'grad_norm': 26.79128646850586, 'learning_rate': 1.7698339834299064e-05, 'epoch': 0.89}


                                               
 33%|███▎      | 22/66 [02:11<01:05,  1.49s/it]

...

{'loss': 0.0324, 'grad_norm': 8.140250205993652, 'learning_rate': 5.060239153161872e-07, 'epoch': 2.67}


                                               
 33%|███▎      | 22/66 [03:32<01:05,  1.49s/it]

{'eval_bge-base-zh-v1.5_cosine_accuracy@1': 0.7014925373134329, 'eval_bge-base-zh-v1.5_cosine_accuracy@3': 0.8208955223880597, 'eval_bge-base-zh-v1.5_cosine_accuracy@5': 0.8507462686567164, 'eval_bge-base-zh-v1.5_cosine_accuracy@10': 0.8656716417910447, 'eval_bge-base-zh-v1.5_cosine_precision@1': 0.7014925373134329, 'eval_bge-base-zh-v1.5_cosine_precision@3': 0.2736318407960198, 'eval_bge-base-zh-v1.5_cosine_precision@5': 0.17014925373134326, 'eval_bge-base-zh-v1.5_cosine_precision@10': 0.08656716417910446, 'eval_bge-base-zh-v1.5_cosine_recall@1': 0.7014925373134329, 'eval_bge-base-zh-v1.5_cosine_recall@3': 0.8208955223880597, 'eval_bge-base-zh-v1.5_cosine_recall@5': 0.8507462686567164, 'eval_bge-base-zh-v1.5_cosine_recall@10': 0.8656716417910447, 'eval_bge-base-zh-v1.5_cosine_ndcg@10': 0.7862251404707558, 'eval_bge-base-zh-v1.5_cosine_mrr@10': 0.7601368159203978, 'eval_bge-base-zh-v1.5_cosine_map@100': 0.7629992332622932, 'eval_runtime': 4.877, 'eval_samples_per_second': 0.0, 'eval_st

                                               
100%|██████████| 66/66 [01:59<00:00,  1.81s/it]


{'train_runtime': 119.3288, 'train_samples_per_second': 4.475, 'train_steps_per_second': 0.553, 'train_loss': 0.14111090564366544, 'epoch': 2.93}
cost time: 122.41s
